# Genesys Spectrogram Dataset - RF Fingerprinting (Models 4-6)

Multi-task RF fingerprinting benchmark on the Genesys Spectrogram Dataset using Xception, ViT-Small, and Swin-Tiny.

## Tasks
| Task | Type | Classes |
|------|------|---------|
| Device Identification | Multiclass | 7 (uav1..uav7) |
| Distance Estimation | Multiclass | 4 (6ft, 9ft, 12ft, 15ft) |

## Data Split Strategy
- 80/20 stratified burst-level split (prevents leakage across SNR conditions)
- All SNR levels used for training; per-SNR evaluation on held-out 20%
- Deterministic split with random_state=42


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import re
import json
import pickle
import shutil
import math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from datetime import datetime
from collections import Counter
from copy import deepcopy

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import timm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torchvision.transforms as transforms

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f"Device      : {device}")
print(f"GPUs        : {num_gpus}")
if torch.cuda.is_available():
    for i in range(num_gpus):
        print(f"  GPU {i}     : {torch.cuda.get_device_name(i)}")
use_multi_gpu = num_gpus > 1


In [ ]:
@dataclass
class Config:
    # Input
    img_size: int = 224
    in_channels: int = 3

    # Tasks
    num_devices: int = 7        # uav1 .. uav7
    num_distances: int = 4      # 6ft, 9ft, 12ft, 15ft

    # Training
    batch_size: int = 32
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    epochs: int = 60
    patience: int = 15
    grad_clip_norm: float = 1.0
    label_smoothing: float = 0.1

    # Paths
    data_root: str = "/kaggle/input/datasets/sambhavnayak/genesys-spectrogram-dataset"

    # SNR folders present in each UAV directory
    snr_levels: Tuple[str, ...] = ("clean", "snr_20dB", "snr_15dB", "snr_10dB", "snr_5dB", "snr_0dB")

    # Models to train
    models: List[str] = field(default_factory=lambda: ["xception", "vit_small", "swin_tiny"])


cfg = Config()
TASKS = ["device_id", "distance"]

DEVICE_MAP = {f"uav{i}": i - 1 for i in range(1, 8)}     # uav1->0 ... uav7->6
DISTANCE_MAP = {"6ft": 0, "9ft": 1, "12ft": 2, "15ft": 3}

print("Config initialized")
print(f"  Batch size : {cfg.batch_size}")
print(f"  Epochs     : {cfg.epochs}  Patience: {cfg.patience}")
print(f"  Devices    : {cfg.num_devices}  Distances: {cfg.num_distances}")
print(f"  Models     : {cfg.models}")
print(f"  SNR levels : {cfg.snr_levels}")


## Data Loading

Filename pattern: `{uavID}_{distance}_{burstID}_{frameID}.png`  
Example: `uav1_12ft_burst1_1.png`

Split is performed at the **burst level** — all frames from the same physical burst (across all SNR conditions) are assigned to either train or validation, never both. This prevents temporal leakage.

In [ ]:
def parse_filename(fname: str) -> Tuple[str, str, str]:
    """
    Returns (uav_id_str, distance_str, burst_id_str).
    Pattern: uav{N}_{dist}_{burstID}_{frameID}.png
    """
    stem = Path(fname).stem                           # e.g. uav1_12ft_burst1_1
    parts = stem.split('_')
    uav_id   = parts[0]                              # uav1
    distance = parts[1]                              # 12ft
    burst_id = parts[2]                              # burst1
    return uav_id, distance, burst_id


def scan_dataset(data_root: str, snr_levels: Tuple[str, ...]) -> List[Dict]:
    """
    Walks the dataset tree and returns a list of records.
    Each record: {path, device_label, distance_label, snr, uav_id, burst_key}
    burst_key = (uav_id, distance, burst_id) -- used for leakage-free split.
    """
    records = []
    root = Path(data_root)
    for uav_dir in sorted(root.iterdir()):
        if not uav_dir.is_dir() or not uav_dir.name.startswith('uav'):
            continue
        for snr in snr_levels:
            snr_dir = uav_dir / snr
            if not snr_dir.exists():
                continue
            for img_path in sorted(snr_dir.glob('*.png')):
                try:
                    uav_id, distance, burst_id = parse_filename(img_path.name)
                    if uav_id not in DEVICE_MAP or distance not in DISTANCE_MAP:
                        continue
                    records.append({
                        'path'          : str(img_path),
                        'device_label'  : DEVICE_MAP[uav_id],
                        'distance_label': DISTANCE_MAP[distance],
                        'snr'           : snr,
                        'uav_id'        : uav_id,
                        'burst_key'     : (uav_id, distance, burst_id),
                    })
                except Exception:
                    continue
    return records


print("Scanning dataset...")
all_records = scan_dataset(cfg.data_root, cfg.snr_levels)
print(f"Total images found : {len(all_records)}")

# Distribution check
snr_counts = Counter(r['snr'] for r in all_records)
uav_counts = Counter(r['device_label'] for r in all_records)
print("\nPer-SNR image counts:")
for snr, cnt in sorted(snr_counts.items()):
    print(f"  {snr:<12}: {cnt:,}")
print("\nPer-device image counts:")
for dev, cnt in sorted(uav_counts.items()):
    print(f"  uav{dev+1}       : {cnt:,}")


In [ ]:
def burst_level_split(
    records: List[Dict],
    val_fraction: float = 0.2,
    random_state: int = 42
) -> Tuple[List[Dict], List[Dict]]:
    """
    Splits at the burst level (uav_id, distance, burst_id) to prevent data leakage.
    Stratified by device_id so class balance is maintained across splits.
    All SNR conditions for a given burst go to the same partition.
    """
    # Collect unique bursts and their device label
    burst_to_device = {}
    for r in records:
        bk = r['burst_key']
        burst_to_device[bk] = r['device_label']

    burst_keys = sorted(burst_to_device.keys())
    burst_labels = [burst_to_device[bk] for bk in burst_keys]

    train_keys, val_keys = train_test_split(
        burst_keys,
        test_size=val_fraction,
        stratify=burst_labels,
        random_state=random_state,
    )

    train_set = set(map(tuple, train_keys))
    val_set   = set(map(tuple, val_keys))

    train_records = [r for r in records if tuple(r['burst_key']) in train_set]
    val_records   = [r for r in records if tuple(r['burst_key']) in val_set]

    return train_records, val_records


train_records, val_records = burst_level_split(all_records, val_fraction=0.2, random_state=42)

print(f"Train images : {len(train_records):,}")
print(f"Val images   : {len(val_records):,}")
print(f"Split ratio  : {len(train_records)/(len(train_records)+len(val_records)):.1%} / {len(val_records)/(len(train_records)+len(val_records)):.1%}")

# Verify no burst key overlap between splits
train_bursts = {tuple(r['burst_key']) for r in train_records}
val_bursts   = {tuple(r['burst_key']) for r in val_records}
overlap = train_bursts & val_bursts
print(f"Burst overlap (must be 0): {len(overlap)}")

# Per-SNR split distribution
print("\nPer-SNR distribution in val split:")
val_snr_counts = Counter(r['snr'] for r in val_records)
for snr in cfg.snr_levels:
    print(f"  {snr:<12}: {val_snr_counts.get(snr, 0):,} images")


In [ ]:
class GenesysDataset(Dataset):
    """RF Spectrogram Dataset for device identification and distance estimation."""

    def __init__(self, records: List[Dict], transform=None):
        self.records   = records
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        r   = self.records[idx]
        img = Image.open(r['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = {
            'device_id': torch.tensor(r['device_label'],   dtype=torch.long),
            'distance' : torch.tensor(r['distance_label'], dtype=torch.long),
        }
        return img, labels


def get_train_transforms(cfg: Config) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def get_val_transforms(cfg: Config) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build_dataloaders(
    cfg: Config,
    train_records: List[Dict],
    val_records: List[Dict]
) -> Tuple[DataLoader, DataLoader]:
    """Builds train/val DataLoaders. Training uses WeightedRandomSampler for device balance."""
    train_ds = GenesysDataset(train_records, transform=get_train_transforms(cfg))
    val_ds   = GenesysDataset(val_records,   transform=get_val_transforms(cfg))

    # Weight by device_id to counter any class imbalance
    device_labels = [r['device_label'] for r in train_records]
    class_counts  = Counter(device_labels)
    weights       = [1.0 / class_counts[lb] for lb in device_labels]
    sampler       = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, sampler=sampler,
        num_workers=4, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=4, pin_memory=True,
    )
    return train_loader, val_loader


train_loader, val_loader = build_dataloaders(cfg, train_records, val_records)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


## Loss Function & Multi-Task Head

Homoscedastic uncertainty weighting (Kendall et al., 2018) automatically balances the device identification and distance estimation tasks during training.

In [ ]:
class HomoscedasticMultiTaskLoss(nn.Module):
    """Homoscedastic uncertainty-weighted multi-task loss (Kendall et al., 2018)."""

    def __init__(self, task_names: List[str], label_smoothing: float = 0.1):
        super().__init__()
        self.task_names = task_names
        self.log_vars   = nn.ParameterDict({
            t: nn.Parameter(torch.zeros(1)) for t in task_names
        })
        self.criteria   = nn.ModuleDict({
            t: nn.CrossEntropyLoss(label_smoothing=label_smoothing) for t in task_names
        })

    def forward(
        self,
        logits: Dict[str, torch.Tensor],
        labels: Dict[str, torch.Tensor],
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        losses = {}
        total  = torch.tensor(0.0, device=next(iter(logits.values())).device)
        for t in self.task_names:
            if t not in logits or t not in labels:
                continue
            ce        = self.criteria[t](logits[t], labels[t])
            precision = torch.exp(-self.log_vars[t])
            total     = total + 0.5 * precision * ce + 0.5 * self.log_vars[t]
            losses[t] = ce.item()
        losses['total'] = total.item()
        return total, losses


class MultiTaskHead(nn.Module):
    """Shared classification head: device identification + distance estimation."""

    def __init__(self, in_features: int, num_devices: int, num_distances: int, dropout: float = 0.3):
        super().__init__()
        self.dropout       = nn.Dropout(dropout)
        self.head_device   = nn.Linear(in_features, num_devices)
        self.head_distance = nn.Linear(in_features, num_distances)

    def forward(self, features: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = self.dropout(features)
        return {
            'device_id': self.head_device(x),
            'distance' : self.head_distance(x),
        }


print("HomoscedasticMultiTaskLoss and MultiTaskHead defined")


## Model 4: Xception (Depth-Separable CNN)

- Depthwise separable convolutions decouple spatial (frequency-time) and channel-wise RF learning
- 36 convolutional layers with residual connections
- Feature dim: 2048 -> MultiTaskHead

In [ ]:
class XceptionMultiTask(nn.Module):
    """Xception backbone with depthwise separable convolutions."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('xception', pretrained=pretrained, num_classes=0)
        self.head      = MultiTaskHead(2048, cfg.num_devices, cfg.num_distances)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        return self.head(self.backbone(x))


_m = XceptionMultiTask(pretrained=False)
_x = torch.randn(2, 3, 224, 224)
_o = _m(_x)
print(f"XceptionMultiTask  | params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"  output device_id : {_o['device_id'].shape}  distance: {_o['distance'].shape}")
del _m, _x, _o


## Model 5: Vision Transformer (ViT-Small/16)

- Global self-attention captures long-range frequency dependencies
- 16x16 patches -> 196 tokens encoding local frequency-time regions
- Feature dim: 384 -> MultiTaskHead

In [ ]:
class ViTSmallMultiTask(nn.Module):
    """ViT-Small/16 with global self-attention for RF spectrogram analysis."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('vit_small_patch16_224', pretrained=pretrained, num_classes=0)
        self.head      = MultiTaskHead(384, cfg.num_devices, cfg.num_distances)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        return self.head(self.backbone(x))


_m = ViTSmallMultiTask(pretrained=False)
_x = torch.randn(2, 3, 224, 224)
_o = _m(_x)
print(f"ViTSmallMultiTask  | params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"  output device_id : {_o['device_id'].shape}  distance: {_o['distance'].shape}")
del _m, _x, _o


## Model 6: Swin Transformer Tiny

- Shifted window attention balances local RF pattern extraction with cross-region interaction
- Hierarchical feature maps (56->28->14->7) capture multi-scale frequency structures
- Feature dim: 768 -> MultiTaskHead

In [ ]:
class SwinTinyMultiTask(nn.Module):
    """Swin-Tiny with shifted window attention for hierarchical RF pattern extraction."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, num_classes=0)
        self.head      = MultiTaskHead(768, cfg.num_devices, cfg.num_distances)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        return self.head(self.backbone(x))


_m = SwinTinyMultiTask(pretrained=False)
_x = torch.randn(2, 3, 224, 224)
_o = _m(_x)
print(f"SwinTinyMultiTask  | params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"  output device_id : {_o['device_id'].shape}  distance: {_o['distance'].shape}")
del _m, _x, _o


In [ ]:
MODEL_REGISTRY = {
    'xception' : XceptionMultiTask,
    'vit_small': ViTSmallMultiTask,
    'swin_tiny': SwinTinyMultiTask,
}

print(f"Model registry: {list(MODEL_REGISTRY.keys())}")


## Training Infrastructure

In [ ]:
def adjust_learning_rate(optimizer, epoch: int, cfg: Config) -> float:
    """Linear warmup for 5 epochs, then cosine decay."""
    warmup_epochs = 5
    if epoch < warmup_epochs:
        lr = cfg.learning_rate * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / max(cfg.epochs - warmup_epochs, 1)
        lr = cfg.learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr


print("Learning rate scheduler defined  (5 epoch warmup + cosine decay)")


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    cfg: Config,
) -> Tuple[float, Dict[str, float]]:
    """One training epoch with mixed precision and gradient clipping."""
    model.train()
    total_loss    = 0.0
    task_correct  = {t: 0 for t in TASKS}
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = {k: v.to(device, non_blocking=True) for k, v in labels.items()}

        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            logits    = model(images)
            loss, _   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_samples += bs
        for t in TASKS:
            task_correct[t] += (logits[t].argmax(1) == labels[t]).sum().item()

    avg_loss = total_loss / total_samples
    avg_acc  = {t: task_correct[t] / total_samples * 100.0 for t in TASKS}
    return avg_loss, avg_acc


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
) -> Tuple[float, Dict]:
    """Validation pass returning loss and per-task accuracy/F1."""
    model.eval()
    total_loss    = 0.0
    all_preds     = {t: [] for t in TASKS}
    all_labels    = {t: [] for t in TASKS}
    total_samples = 0

    for images, labels in loader:
        images     = images.to(device, non_blocking=True)
        labels_dev = {k: v.to(device, non_blocking=True) for k, v in labels.items()}
        with autocast(dtype=torch.float16):
            logits  = model(images)
            loss, _ = criterion(logits, labels_dev)
        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_samples += bs
        for t in TASKS:
            all_preds[t].extend(logits[t].argmax(1).cpu().tolist())
            all_labels[t].extend(labels[t].tolist())

    metrics = {}
    for t in TASKS:
        yt = np.array(all_labels[t])
        yp = np.array(all_preds[t])
        metrics[t] = {
            'accuracy': accuracy_score(yt, yp) * 100.0,
            'f1_score': f1_score(yt, yp, average='weighted', zero_division=0) * 100.0,
        }
    metrics['average'] = {'accuracy': np.mean([metrics[t]['accuracy'] for t in TASKS])}
    return total_loss / total_samples, metrics


print("train_one_epoch and validate defined")


In [ ]:
def train_model(
    model_name: str,
    model_class,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: Config,
) -> Dict:
    """Full training loop with early stopping, mixed precision, and cosine LR."""
    print(f"\n{'='*70}")
    print(f"Training: {model_name.upper()}")
    print(f"  Epochs: {cfg.epochs}  Patience: {cfg.patience}  Batch: {cfg.batch_size}")
    print(f"{'='*70}")

    model = model_class(pretrained=True)
    if use_multi_gpu:
        model = nn.DataParallel(model)
        print(f"  DataParallel on {num_gpus} GPUs")
    model = model.to(device)

    criterion = HomoscedasticMultiTaskLoss(
        task_names=TASKS,
        label_smoothing=cfg.label_smoothing,
    ).to(device)

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(criterion.parameters()),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scaler = GradScaler()

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc' : [], 'val_acc' : [],
        'lr'        : [],
    }
    best_val_acc    = 0.0
    best_metrics    = None
    patience_counter = 0

    for epoch in range(cfg.epochs):
        lr = adjust_learning_rate(optimizer, epoch, cfg)
        history['lr'].append(lr)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, cfg)
        val_loss, val_metrics = validate(model, val_loader, criterion)

        train_avg = np.mean(list(train_acc.values()))
        val_avg   = val_metrics['average']['accuracy']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_avg)
        history['val_acc'].append(val_avg)

        improved = val_avg > best_val_acc
        if improved:
            best_val_acc = val_avg
            best_metrics = deepcopy(val_metrics)
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 5 == 0 or epoch == cfg.epochs - 1 or improved:
            lv_str = " ".join(f"{t}:{criterion.log_vars[t].item():.3f}" for t in TASKS)
            print(
                f"Epoch {epoch+1:3d}/{cfg.epochs}"
                f" | LR {lr:.6f}"
                f" | Train Loss {train_loss:.4f}  Acc {train_avg:.2f}%"
                f" | Val Loss {val_loss:.4f}  Acc {val_avg:.2f}%"
                f" | Best {best_val_acc:.2f}%"
                f" | LogVar [{lv_str}]"
            )

        if patience_counter >= cfg.patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    # Capture final model state after training ends
    core = model.module if isinstance(model, nn.DataParallel) else model
    final_state = deepcopy(core.state_dict())

    print(f"\nTraining complete. Best val acc: {best_val_acc:.2f}%")
    print(f"  {'Task':<12} {'Acc':>8} {'F1':>8}")
    print(f"  {'-'*30}")
    for t in TASKS:
        m = best_metrics[t]
        print(f"  {t:<12} {m['accuracy']:>7.2f}% {m['f1_score']:>7.2f}%")

    del model, criterion, optimizer, scaler
    torch.cuda.empty_cache()

    return {
        'model_name'  : model_name,
        'best_val_acc': best_val_acc,
        'best_metrics': best_metrics,
        'final_state' : final_state,
        'history'     : history,
    }


print("train_model defined")


## Training Execution

Training Xception, ViT-Small, and Swin-Tiny with the Genesys multitask protocol.

In [ ]:
all_results = {}

for model_name in cfg.models:
    results = train_model(
        model_name,
        MODEL_REGISTRY[model_name],
        train_loader,
        val_loader,
        cfg,
    )
    all_results[model_name] = results

print("\n" + "="*70)
print("ALL MODELS TRAINED")
print("="*70)


In [ ]:
print("="*80)
print("TRAINING RESULTS SUMMARY")
print("="*80)
print(f"\n{'Model':<20} {'device_id Acc':>15} {'distance Acc':>14} {'Avg Val Acc':>13}")
print("-"*65)
for name, res in all_results.items():
    m   = res['best_metrics']
    avg = res['best_val_acc']
    print(f"{name:<20} {m['device_id']['accuracy']:>14.2f}% {m['distance']['accuracy']:>13.2f}% {avg:>12.2f}%")

best_name = max(all_results, key=lambda n: all_results[n]['best_val_acc'])
print(f"\nBest model: {best_name}  ({all_results[best_name]['best_val_acc']:.2f}%)")


In [ ]:
fig, axes = plt.subplots(1, len(all_results), figsize=(6 * len(all_results), 5))
if len(all_results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, all_results.items()):
    h      = res['history']
    epochs = range(1, len(h['val_acc']) + 1)
    ax.plot(epochs, h['train_loss'], label='Train Loss', alpha=0.7)
    ax.plot(epochs, h['val_loss'],   label='Val Loss',   linewidth=2)
    ax2 = ax.twinx()
    ax2.plot(epochs, h['train_acc'], '--', color='C2', label='Train Acc', alpha=0.7)
    ax2.plot(epochs, h['val_acc'],   '--', color='C3', label='Val Acc',   linewidth=2)
    ax.set_title(f"{name}\nBest: {res['best_val_acc']:.1f}%")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_ylim([0, 100])
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("Training curves saved to training_curves.png")


## Evaluation

### Per-SNR Metrics

Each model is evaluated on the held-out 20% validation split filtered by SNR level.
Metrics: Accuracy, F1 (weighted), FPR (macro), FNR (macro).

In [ ]:
TASK_NUM_CLASSES = {
    'device_id': cfg.num_devices,
    'distance' : cfg.num_distances,
}


def compute_task_metrics(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> Dict:
    """Accuracy, weighted F1, macro-FPR, macro-FNR for one task."""
    acc = accuracy_score(y_true, y_pred) * 100.0
    f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100.0
    cm  = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    fpr_list, fnr_list = [], []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
        fnr_list.append(fn / (fn + tp) if (fn + tp) > 0 else 0.0)
    return {
        'accuracy': acc,
        'f1'      : f1,
        'fpr'     : np.mean(fpr_list) * 100.0,
        'fnr'     : np.mean(fnr_list) * 100.0,
    }


@torch.no_grad()
def evaluate_records(
    model: nn.Module,
    records: List[Dict],
    cfg: Config,
) -> Dict:
    """Evaluate model on a list of records and return per-task metrics."""
    if not records:
        return None
    ds     = GenesysDataset(records, transform=get_val_transforms(cfg))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=4, pin_memory=True)
    all_preds  = {t: [] for t in TASKS}
    all_labels = {t: [] for t in TASKS}
    model.eval()
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with autocast(dtype=torch.float16):
            logits = model(images)
        for t in TASKS:
            all_preds[t].extend(logits[t].argmax(1).cpu().tolist())
            all_labels[t].extend(labels[t].tolist())
    result = {}
    for t in TASKS:
        result[t] = compute_task_metrics(
            np.array(all_labels[t]),
            np.array(all_preds[t]),
            TASK_NUM_CLASSES[t],
        )
    result['average'] = {
        'accuracy': np.mean([result[t]['accuracy'] for t in TASKS]),
        'f1'      : np.mean([result[t]['f1']       for t in TASKS]),
        'fpr'     : np.mean([result[t]['fpr']       for t in TASKS]),
        'fnr'     : np.mean([result[t]['fnr']       for t in TASKS]),
    }
    return result


def load_best_model(model_name: str, state_dict: dict) -> nn.Module:
    model = MODEL_REGISTRY[model_name](pretrained=False)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model


def print_snr_metrics(snr_label: str, n_samples: int, result: Dict):
    """Pretty-print per-SNR metrics for all tasks."""
    print(f"\n  SNR: {snr_label}  (n={n_samples:,})")
    print(f"  {'Task':<14} {'Accuracy':>10} {'F1':>10} {'FPR':>10} {'FNR':>10}")
    print(f"  {'-'*56}")
    for t in TASKS:
        m = result[t]
        print(f"  {t:<14} {m['accuracy']:>9.2f}% {m['f1']:>9.2f}% {m['fpr']:>9.2f}% {m['fnr']:>9.2f}%")
    a = result['average']
    print(f"  {'AVERAGE':<14} {a['accuracy']:>9.2f}% {a['f1']:>9.2f}% {a['fpr']:>9.2f}% {a['fnr']:>9.2f}%")


print("Evaluation utilities defined")


In [ ]:
# Pre-bucket val_records by SNR for fast lookup
val_by_snr = {snr: [] for snr in cfg.snr_levels}
for r in val_records:
    val_by_snr[r['snr']].append(r)

noise_results = {}

print("=" * 80)
print("DETAILED PER-SNR EVALUATION")
print("=" * 80)

for model_name, res in all_results.items():
    model = load_best_model(model_name, res['final_state'])
    noise_results[model_name] = {}

    print(f"\n{'='*80}")
    print(f"MODEL: {model_name.upper()}")
    print(f"{'='*80}")

    for snr in cfg.snr_levels:
        records_snr = val_by_snr.get(snr, [])
        result      = evaluate_records(model, records_snr, cfg)
        if result is None:
            print(f"\n  SNR: {snr}  -- no data found, skipping")
            continue
        noise_results[model_name][snr] = result
        print_snr_metrics(snr, len(records_snr), result)

    del model
    torch.cuda.empty_cache()

print("\nEvaluation complete")


In [ ]:
print("=" * 80)
print("OVERALL METRICS (FULL VALIDATION SET)")
print("=" * 80)

for model_name, res in all_results.items():
    model   = load_best_model(model_name, res['final_state'])
    overall = evaluate_records(model, val_records, cfg)
    del model
    torch.cuda.empty_cache()

    print(f"\nModel: {model_name.upper()}")
    print(f"  {'Task':<14} {'Accuracy':>10} {'F1':>10} {'FPR':>10} {'FNR':>10}")
    print(f"  {'-'*56}")
    for t in TASKS:
        m = overall[t]
        print(f"  {t:<14} {m['accuracy']:>9.2f}% {m['f1']:>9.2f}% {m['fpr']:>9.2f}% {m['fnr']:>9.2f}%")
    a = overall['average']
    print(f"  {'AVERAGE':<14} {a['accuracy']:>9.2f}% {a['f1']:>9.2f}% {a['fpr']:>9.2f}% {a['fnr']:>9.2f}%")


In [ ]:
print("=" * 90)
print("NOISE ROBUSTNESS SUMMARY -- Device Identification Accuracy")
print("=" * 90)
snr_short = [s.replace('snr_', '') for s in cfg.snr_levels]
print(f"\n{'Model':<22}" + "".join(f" {s:>9}" for s in snr_short))
print("-" * 80)
for model_name in all_results:
    row = f"{model_name:<22}"
    for snr in cfg.snr_levels:
        if snr in noise_results.get(model_name, {}):
            acc = noise_results[model_name][snr]['device_id']['accuracy']
            row += f" {acc:>8.1f}%"
        else:
            row += f"  {'N/A':>8}"
    print(row)

print()
print("=" * 90)
print("NOISE ROBUSTNESS SUMMARY -- Distance Estimation Accuracy")
print("=" * 90)
print(f"\n{'Model':<22}" + "".join(f" {s:>9}" for s in snr_short))
print("-" * 80)
for model_name in all_results:
    row = f"{model_name:<22}"
    for snr in cfg.snr_levels:
        if snr in noise_results.get(model_name, {}):
            acc = noise_results[model_name][snr]['distance']['accuracy']
            row += f" {acc:>8.1f}%"
        else:
            row += f"  {'N/A':>8}"
    print(row)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
snr_labels_plot = [s.replace('snr_', '') for s in cfg.snr_levels]
colors = plt.cm.tab10(np.linspace(0, 1, len(all_results)))

for ax_idx, task in enumerate(['device_id', 'distance']):
    ax = axes[ax_idx]
    for idx, model_name in enumerate(all_results):
        accs = []
        for snr in cfg.snr_levels:
            if snr in noise_results.get(model_name, {}):
                accs.append(noise_results[model_name][snr][task]['accuracy'])
            else:
                accs.append(float('nan'))
        ax.plot(range(len(cfg.snr_levels)), accs, 'o-',
                label=model_name, color=colors[idx], linewidth=2, markersize=8)
    ax.set_title(f"Noise Robustness: {task.replace('_', ' ').title()} Accuracy")
    ax.set_xlabel("SNR Level")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xticks(range(len(cfg.snr_levels)))
    ax.set_xticklabels(snr_labels_plot)
    ax.set_ylim([0, 100])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('noise_robustness.png', dpi=150, bbox_inches='tight')
plt.close()
print("Noise robustness plot saved to noise_robustness.png")


## Save All Artifacts

Saves model weights, training history, noise robustness results, and creates a downloadable ZIP.

In [ ]:
OUTPUT_DIR = 'genesys_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model weights -- one file per model, saved once at the end of training
print("Saving final model weights...")
for model_name, res in all_results.items():
    save_path = f"{OUTPUT_DIR}/{model_name}_final.pth"
    torch.save({
        'model_name'  : model_name,
        'state_dict'  : res['final_state'],
        'best_val_acc': res['best_val_acc'],
    }, save_path)
    print(f"  {model_name}: val_acc={res['best_val_acc']:.2f}%  -> {save_path}")

# Results JSON
results_summary = {
    'timestamp'   : datetime.now().isoformat(),
    'dataset'     : 'Genesys Spectrogram Dataset',
    'tasks'       : TASKS,
    'snr_levels'  : list(cfg.snr_levels),
    'config'      : {
        'batch_size'    : cfg.batch_size,
        'learning_rate' : cfg.learning_rate,
        'epochs'        : cfg.epochs,
        'patience'      : cfg.patience,
        'num_devices'   : cfg.num_devices,
        'num_distances' : cfg.num_distances,
    },
    'models'       : {
        name: {
            'best_val_acc': res['best_val_acc'],
            'per_task'    : {
                t: {
                    'accuracy': res['best_metrics'][t]['accuracy'],
                    'f1_score': res['best_metrics'][t]['f1_score'],
                }
                for t in TASKS
            },
        }
        for name, res in all_results.items()
    },
    'noise_robustness': {
        model_name: {
            snr: {
                t: {k: round(v, 2) for k, v in data.items()}
                for t, data in snr_data.items()
            }
            for snr, snr_data in snr_res.items()
        }
        for model_name, snr_res in noise_results.items()
    },
}

with open(f"{OUTPUT_DIR}/results_summary.json", 'w') as f:
    json.dump(results_summary, f, indent=2)
print("Saved results_summary.json")

# Training histories
with open(f"{OUTPUT_DIR}/training_histories.pkl", 'wb') as f:
    pickle.dump({name: res['history'] for name, res in all_results.items()}, f)
print("Saved training_histories.pkl")

# Noise robustness data
with open(f"{OUTPUT_DIR}/noise_results.pkl", 'wb') as f:
    pickle.dump(noise_results, f)
print("Saved noise_results.pkl")

# Plots
for fname in ['training_curves.png', 'noise_robustness.png']:
    if os.path.exists(fname):
        shutil.copy(fname, f"{OUTPUT_DIR}/{fname}")
        print(f"Saved {fname}")

# ZIP
zip_name = f"genesys_models_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
print(f"\nZIP created: {zip_name}.zip")
print(f"Contents   : {os.listdir(OUTPUT_DIR)}")

try:
    from IPython.display import FileLink, display
    display(FileLink(f"{zip_name}.zip"))
except ImportError:
    print(f"Download: {os.path.abspath(zip_name)}.zip")
